# Synthetic Control Panel

Этот notebook работает **напрямую через Python API** из [`synthetic_api.py`](synthetic_api.py), без `subprocess` и без CLI-оберток.

Идея такая:
1. по отдельности задать конфиги этапов;
2. по шагам материализовать синтетический сценарий;
3. собрать `Scenario` вручную или одной функцией;
4. передать готовый сценарий в массив методов;
5. получить таблицы, графики и свипы по параметрам.

То есть notebook здесь — именно **панель управления исследованием**, а `synthetic_api.py` — **серый ящик с этапами и пайплайном**.


In [18]:
from dataclasses import replace
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'synthetic_api.py').exists():
    PROJECT_DIR = Path('/Users/karimau/Projects_C++/Course Project')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import synthetic_api as api

PROJECT_DIR


PosixPath('/Users/karimau/Projects_C++/Course Project')

## 1. Конфиги этапов

Каждый этап задается отдельно. Это главный принцип архитектуры: можно менять только один слой, не трогая остальные.


In [19]:
matrix_cfg = api.MatrixConfig(
    m=60,
    n=60,
    rank=4,
    factor_distribution='gaussian',
    singular_value_profile='linear',
    singular_scale=1.0,
    coherence_mode='incoherent',
    coherence_strength=0.9,
)

structure_cfg = api.StructureConfig(
    mode='none',
    strength=0.0,
    secondary_strength=0.0,
)

missingness_field_cfg = api.MissingnessFieldConfig(
    mode='random',
    concentration=1.0,
    width=0.15,
    value_dependence=0.0,
)

mask_sampling_cfg = api.MaskSamplingConfig(
    observed_fraction=0.35,
    exact_fraction=True,
    threshold=None,
    invert_scores=False,
)

split_cfg = api.SplitConfig(
    validation_fraction=0.15,
    test_fraction=0.15,
)

noise_cfg = api.NoiseConfig(
    mode='gaussian',
    std=0.02,
    outlier_fraction=0.0,
    outlier_scale=0.0,
)

base_config = api.ScenarioConfig(
    matrix=matrix_cfg,
    structure=structure_cfg,
    missingness_field=missingness_field_cfg,
    mask_sampling=mask_sampling_cfg,
    split=split_cfg,
    noise=noise_cfg,
)

base_config


ScenarioConfig(matrix=MatrixConfig(m=60, n=60, rank=4, factor_distribution='gaussian', singular_value_profile='linear', singular_scale=1.0, coherence_mode='incoherent', coherence_strength=0.9, sparse_additive_fraction=0.0, sparse_additive_scale=0.0), structure=StructureConfig(mode='none', strength=0.0, secondary_strength=0.0), missingness_field=MissingnessFieldConfig(mode='random', concentration=1.0, width=0.15, row_index=None, col_index=None, value_dependence=0.0), mask_sampling=MaskSamplingConfig(observed_fraction=0.35, exact_fraction=True, threshold=None, invert_scores=False), split=SplitConfig(validation_fraction=0.15, test_fraction=0.15), noise=NoiseConfig(mode='gaussian', std=0.02, outlier_fraction=0.0, outlier_scale=0.0))

## 2. Поэтапная материализация сценария

Ниже сценарий собирается **вручную по этапам**. Это удобно, когда нужно отдельно смотреть на:
- истинную матрицу;
- структурную модификацию;
- поле пропусков;
- итоговую маску;
- разбиение `fit / validation / test`.


In [20]:
seed = 42
rng = api.make_rng(seed)

X_true = api.generate_base_matrix(matrix_cfg, rng)
X_structured = api.apply_structure(X_true, structure_cfg, rng)
missingness_field = api.build_missingness_field(X_structured, missingness_field_cfg, rng)
observed_mask = api.sample_observed_mask(missingness_field, mask_sampling_cfg, rng)
mask_split = api.split_observed_mask(observed_mask, split_cfg, rng)
Y_observed = api.build_observed_matrix(X_structured, mask_split.observed_mask, noise_cfg, rng)

print('X_true shape:', X_true.shape)
print('observed fraction:', observed_mask.mean())
print('fit fraction:', mask_split.fit_mask.mean())
print('validation fraction:', mask_split.validation_mask.mean())
print('test fraction:', mask_split.test_mask.mean())


X_true shape: (60, 60)
observed fraction: 0.35
fit fraction: 0.25277777777777777
validation fraction: 0.04472222222222222
test fraction: 0.0525


In [21]:
scenario = api.package_scenario(
    config=base_config,
    seed=seed,
    X_true=X_true,
    X_structured=X_structured,
    missingness_field=missingness_field,
    mask_split=mask_split,
    Y_observed=Y_observed,
    metadata={'assembly_mode': 'manual_stage_pipeline'},
)

scenario.metadata


{'observed_fraction_effective': 0.35,
 'train_fraction_effective': 0.2975,
 'fit_fraction_effective': 0.25277777777777777,
 'validation_fraction_effective': 0.04472222222222222,
 'test_fraction_effective': 0.0525,
 'assembly_mode': 'manual_stage_pipeline'}

## 3. Быстрая визуальная проверка этапов

Этот блок нужен именно для исследования: прежде чем запускать методы, можно глазами проверить, что именно мы сгенерировали.


In [22]:
api.plot_matrix(X_true, title='True low-rank matrix');
api.plot_matrix(X_structured, title='Structured matrix');
api.plot_matrix(missingness_field, title='Missingness field');
api.plot_mask(mask_split.observed_mask, title='Observed mask');
api.plot_mask(mask_split.fit_mask, title='Fit mask');
api.plot_mask(mask_split.validation_mask, title='Validation mask');
api.plot_mask(mask_split.test_mask, title='Test mask');

# Анализ спектра матрицы
sv_true = api.singular_values(X_true)
sv_structured = api.singular_values(X_structured)
print('Первые сингулярные значения X_true:', sv_true[:10])
print('Первые сингулярные значения X_structured:', sv_structured[:10])
api.plot_singular_values(X_true, title='Singular values of X_true');
api.plot_singular_values(X_true, title='Singular values of X_true (log scale)', log_scale=True);
api.plot_singular_values(X_structured, title='Singular values of X_structured');
api.plot_singular_values(X_structured, title='Singular values of X_structured (log scale)', log_scale=True);


## 4. Набор методов

Методы задаются как обычный Python-массив. Тут нет preset-магии: мы явно контролируем, что именно сравниваем.


In [23]:
methods = [
    api.make_method('soft_impute', rank=matrix_cfg.rank, max_iter=80, lambda_scale=0.15, label='Soft-Impute'),
    api.make_method('als', rank=matrix_cfg.rank, max_iter=50, reg=1e-3, init='spectral', label='ALS'),
    api.make_method('rgd', rank=matrix_cfg.rank, max_iter=120, init='spectral', label='RGD'),
    api.make_method('rgd_l2', rank=matrix_cfg.rank, max_iter=120, init='spectral', l2_reg=0.03, label='RGD + L2'),
    api.make_method(
        'compact_l2_selected',
        rank=matrix_cfg.rank,
        max_iter=90,
        init='spectral',
        l2_grid=[1e-3, 3e-3, 1e-2, 3e-2],
        label='Compact RGD + selected L2',
    ),
]

methods


[MethodConfig(name='soft_impute', params={'rank': 4, 'max_iter': 80, 'lambda_scale': 0.15}, label='Soft-Impute'),
 MethodConfig(name='als', params={'rank': 4, 'max_iter': 50, 'reg': 0.001, 'init': 'spectral'}, label='ALS'),
 MethodConfig(name='rgd', params={'rank': 4, 'max_iter': 120, 'init': 'spectral'}, label='RGD'),
 MethodConfig(name='rgd_l2', params={'rank': 4, 'max_iter': 120, 'init': 'spectral', 'l2_reg': 0.03}, label='RGD + L2'),
 MethodConfig(name='compact_l2_selected', params={'rank': 4, 'max_iter': 90, 'init': 'spectral', 'l2_grid': [0.001, 0.003, 0.01, 0.03]}, label='Compact RGD + selected L2')]

## 5. Один прогон на уже собранном `Scenario`

Здесь мы не передаем конфиг в общий benchmark. Мы именно берем **готовый сценарий** и прогоняем на нем методы.


In [24]:
single_run = api.run_experiment_on_scenario(scenario, methods)
records = single_run.records
api.to_dataframe(records)


,seed,method,label,m,n,true_rank,assumed_rank,iterations,runtime_sec,final_train_objective,...,mask_sampling.observed_fraction,mask_sampling.exact_fraction,mask_sampling.threshold,mask_sampling.invert_scores,split.validation_fraction,split.test_fraction,noise.mode,noise.std,noise.outlier_fraction,noise.outlier_scale
0,42,soft_impute,Soft-Impute,60,60,4,4,80,0.036806,0.453415,...,0.35,True,None,False,0.15,0.15,gaussian,0.02,0.0,0.0
1,42,als,ALS,60,60,4,4,44,0.052304,0.116660,...,0.35,True,None,False,0.15,0.15,gaussian,0.02,0.0,0.0
2,42,rgd,RGD,60,60,4,4,120,0.048191,0.117979,...,0.35,True,None,False,0.15,0.15,gaussian,0.02,0.0,0.0
3,42,rgd_l2,RGD + L2,60,60,4,4,120,0.047215,0.511053,...,0.35,True,None,False,0.15,0.15,gaussian,0.02,0.0,0.0
4,42,compact_l2_selected,Compact RGD + selected L2,60,60,4,4,90,0.011705,0.164686,...,0.35,True,None,False,0.15,0.15,gaussian,0.02,0.0,0.0


In [25]:
summary_by_method = api.aggregate_records(records, by=['method'])
api.to_dataframe(summary_by_method)


,method,n_runs,avg_test_rmse,std_test_rmse,avg_test_mae,std_test_mae,avg_relative_fro_error,std_relative_fro_error,avg_runtime_sec,std_runtime_sec,avg_iterations,std_iterations,avg_train_rmse_observed,std_train_rmse_observed
0,als,1,0.030001,0.0,0.016748,0.0,0.266676,0.0,0.052304,0.0,44.0,0.0,0.014760,0.0
1,compact_l2_selected,1,0.018514,0.0,0.013484,0.0,0.192706,0.0,0.011705,0.0,90.0,0.0,0.014984,0.0
2,rgd,1,0.020338,0.0,0.014255,0.0,0.199844,0.0,0.048191,0.0,120.0,0.0,0.014842,0.0
3,rgd_l2,1,0.024334,0.0,0.017790,0.0,0.259332,0.0,0.047215,0.0,120.0,0.0,0.018075,0.0
4,soft_impute,1,0.038863,0.0,0.027718,0.0,0.385584,0.0,0.036806,0.0,80.0,0.0,0.029098,0.0


In [26]:
api.plot_histories(single_run.method_results, metric_key='train_objective', title='Convergence by method');


## 6. Удобная функция для свипов

Ниже — тонкая обертка для исследования. Она меняет только один параметр, запускает все методы и сразу возвращает raw records и summary.


In [27]:
def run_one_factor_study(base_config, parameter_path, values, methods, seeds=(41, 42, 43)):
    runs = api.one_factor_sweep(
        base_config=base_config,
        parameter_path=parameter_path,
        values=values,
        methods=methods,
        seeds=seeds,
    )
    records = api.collect_records(runs)
    summary = api.aggregate_records(records, by=[parameter_path, 'method'])
    return runs, records, summary


## 7. Пример: свип по рангу

Это уже исследовательский режим: меняем только `matrix.rank`, а весь остальной pipeline остается тем же самым.


In [28]:
rank_values = [4, 6, 8, 10]
rank_runs, rank_records, rank_summary = run_one_factor_study(
    base_config=base_config,
    parameter_path='matrix.rank',
    values=rank_values,
    methods=methods,
    seeds=(41, 42),
)

api.to_dataframe(rank_summary).head()


,matrix.rank,method,n_runs,avg_test_rmse,std_test_rmse,avg_test_mae,std_test_mae,avg_relative_fro_error,std_relative_fro_error,avg_runtime_sec,std_runtime_sec,avg_iterations,std_iterations,avg_train_rmse_observed,std_train_rmse_observed
0,4,als,2,0.026881,0.004412,0.017117,0.000522,0.235956,0.043445,0.047242,0.025572,34.0,14.142136,0.014794,0.000047
1,4,compact_l2_selected,2,0.025553,0.009955,0.017576,0.005786,0.263219,0.099720,0.012816,0.001220,90.0,0.000000,0.016785,0.002547
2,4,rgd,2,0.032228,0.016815,0.021227,0.009861,0.308822,0.154118,0.055814,0.012218,120.0,0.000000,0.016502,0.002348
3,4,rgd_l2,2,0.025466,0.001600,0.018352,0.000795,0.256521,0.003976,0.052043,0.006987,120.0,0.000000,0.018280,0.000289
4,4,soft_impute,2,0.038712,0.000213,0.027799,0.000115,0.377798,0.011012,0.034537,0.001848,80.0,0.000000,0.029707,0.000861


In [29]:
api.plot_metric(
    rank_summary,
    x='matrix.rank',
    y='avg_test_rmse',
    hue='method',
    title='RMSE vs true rank',
    xlabel='True rank',
    ylabel='Average test RMSE',
);


## 8. Пример: свип по типу пропусков

Точно так же можно фиксировать матрицу и менять только геометрию пропусков.


In [30]:
missingness_values = ['random', 'block', 'clustered', 'row_focus', 'column_focus']
miss_runs, miss_records, miss_summary = run_one_factor_study(
    base_config=base_config,
    parameter_path='missingness_field.mode',
    values=missingness_values,
    methods=methods,
    seeds=(51, 52),
)

api.to_dataframe(miss_summary).head()


,missingness_field.mode,method,n_runs,avg_test_rmse,std_test_rmse,avg_test_mae,std_test_mae,avg_relative_fro_error,std_relative_fro_error,avg_runtime_sec,std_runtime_sec,avg_iterations,std_iterations,avg_train_rmse_observed,std_train_rmse_observed
0,block,als,2,0.016253,0.002097,0.012042,0.001243,0.473755,0.123821,0.056650,0.004697,50.0,0.000000,0.015239,0.001382
1,block,compact_l2_selected,2,0.020573,0.006804,0.014919,0.004234,0.584882,0.032416,0.011863,0.000187,90.0,0.000000,0.018009,0.001906
2,block,rgd,2,0.020237,0.007628,0.014876,0.005216,0.588698,0.037137,0.046143,0.002338,120.0,0.000000,0.017066,0.001189
3,block,rgd_l2,2,0.021591,0.004428,0.015217,0.002841,0.593371,0.023261,0.045385,0.004069,120.0,0.000000,0.019208,0.001568
4,block,soft_impute,2,0.040763,0.004067,0.029236,0.001920,0.671166,0.019647,0.020564,0.007514,67.0,18.384776,0.035131,0.002084


In [31]:
api.plot_metric(
    miss_summary,
    x='missingness_field.mode',
    y='avg_test_rmse',
    hue='method',
    title='RMSE by missingness pattern',
    xlabel='Missingness pattern',
    ylabel='Average test RMSE',
);


/Users/karimau/Projects_C++/Course Project/synthetic_api.py:968: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  save_path: str | Path | None = None,


## 9. Пример полностью ручного Python-цикла

Если `one_factor_sweep(...)` кажется слишком абстрактным, можно идти совсем вручную: менять конфиг через `replace(...)` / `apply_overrides(...)` и вызывать pipeline в цикле.


In [32]:
manual_records = []
for coherence_mode in ['incoherent', 'coherent_rows', 'coherent_cols', 'coherent_both']:
    cfg = api.apply_overrides(base_config, {'matrix.coherence_mode': coherence_mode})
    run = api.run_single_experiment(cfg, methods, seed=77)
    manual_records.extend(run.records)

manual_summary = api.aggregate_records(manual_records, by=['matrix.coherence_mode', 'method'])
api.to_dataframe(manual_summary)


,matrix.coherence_mode,method,n_runs,avg_test_rmse,std_test_rmse,avg_test_mae,std_test_mae,avg_relative_fro_error,std_relative_fro_error,avg_runtime_sec,std_runtime_sec,avg_iterations,std_iterations,avg_train_rmse_observed,std_train_rmse_observed
0,coherent_both,als,1,0.036754,0.0,0.019160,0.0,0.799336,0.0,0.060390,0.0,50.0,0.0,0.014410,0.0
1,coherent_both,compact_l2_selected,1,0.026876,0.0,0.016417,0.0,0.785324,0.0,0.011900,0.0,90.0,0.0,0.016741,0.0
2,coherent_both,rgd,1,0.041876,0.0,0.018530,0.0,0.817927,0.0,0.047166,0.0,120.0,0.0,0.015091,0.0
3,coherent_both,rgd_l2,1,0.027406,0.0,0.016844,0.0,0.784787,0.0,0.047218,0.0,120.0,0.0,0.016653,0.0
4,coherent_both,soft_impute,1,0.028255,0.0,0.017547,0.0,0.787935,0.0,0.031916,0.0,80.0,0.0,0.023152,0.0
5,coherent_cols,als,1,0.060974,0.0,0.023573,0.0,1.270148,0.0,0.061681,0.0,50.0,0.0,0.015868,0.0
6,coherent_cols,compact_l2_selected,1,0.027124,0.0,0.017714,0.0,0.629170,0.0,0.011870,0.0,90.0,0.0,0.016850,0.0
7,coherent_cols,rgd,1,0.042273,0.0,0.022572,0.0,0.648977,0.0,0.047048,0.0,120.0,0.0,0.016490,0.0
8,coherent_cols,rgd_l2,1,0.032339,0.0,0.018213,0.0,0.635267,0.0,0.046649,0.0,120.0,0.0,0.017600,0.0
9,coherent_cols,soft_impute,1,0.040950,0.0,0.024817,0.0,0.740605,0.0,0.049092,0.0,80.0,0.0,0.032078,0.0


In [33]:
api.plot_metric(
    manual_summary,
    x='matrix.coherence_mode',
    y='avg_test_rmse',
    hue='method',
    title='RMSE by coherence mode',
    xlabel='Coherence mode',
    ylabel='Average test RMSE',
);


## 10. Экспорт таблиц и результатов

Эта секция нужна уже для курсовой: сохраняем raw records, summary и при необходимости готовую `latex`-таблицу.


In [34]:
output_dir = PROJECT_DIR / 'synthetic_outputs'
output_dir.mkdir(exist_ok=True)

api.write_csv(records, output_dir / 'single_run_records.csv')
api.write_csv(rank_records, output_dir / 'rank_sweep_records.csv')
api.write_csv(rank_summary, output_dir / 'rank_sweep_summary.csv')
api.write_csv(miss_summary, output_dir / 'missingness_summary.csv')
api.write_csv(manual_summary, output_dir / 'coherence_summary.csv')

api.write_latex_table(
    rows=rank_summary,
    columns=[
        ('matrix.rank', 'Rank'),
        ('method', 'Method'),
        ('avg_test_rmse', 'Avg. RMSE'),
        ('avg_runtime_sec', 'Avg. time'),
    ],
    caption='Сравнение методов при разных истинных рангах.',
    label='tab:rank-sweep',
    path=output_dir / 'rank_sweep_table.tex',
)

sorted(output_dir.iterdir())[:10]


[PosixPath('/Users/karimau/Projects_C++/Course Project/synthetic_outputs/coherence_summary.csv'),
 PosixPath('/Users/karimau/Projects_C++/Course Project/synthetic_outputs/missingness_summary.csv'),
 PosixPath('/Users/karimau/Projects_C++/Course Project/synthetic_outputs/rank_sweep_records.csv'),
 PosixPath('/Users/karimau/Projects_C++/Course Project/synthetic_outputs/rank_sweep_summary.csv'),
 PosixPath('/Users/karimau/Projects_C++/Course Project/synthetic_outputs/rank_sweep_table.tex'),
 PosixPath('/Users/karimau/Projects_C++/Course Project/synthetic_outputs/single_run_records.csv')]

## 11. Короткая памятка по workflow

Обычно рабочий цикл такой:

1. меняем один из stage-config блоков;
2. пересобираем сценарий вручную или через `assemble_scenario(...)`;
3. подбираем массив методов;
4. делаем один sanity-check прогон;
5. запускаем `run_one_factor_study(...)` или свой цикл;
6. строим графики и сохраняем таблицы.

Если понадобится, следующим шагом можно добавить сюда уже реальные данные World Bank как отдельный pipeline, но синтетическая часть теперь от него полностью отделена.
